[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Use Re-ID

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/re-ID/blob/main/docs/cookbooks/how-to-use-reid.ipynb)

`reid` turns a crop of a person into an appearance embedding: a vector that is close for crops of the same person and far apart for different people. In this notebook we use MOT17, a multi-object tracking benchmark: we load a model, embed the pedestrians in one frame, match them to a frame 30 frames later, and evaluate the model on the comparison a tracker makes.

## Setup

### Check GPU availability

The model runs on every person crop, so use a GPU runtime. If the cell below fails, go to `Runtime` → `Change runtime type` and pick a GPU.

In [ ]:
!nvidia-smi

### Install `reid`

In [ ]:
!pip install -q reid

### Imports

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
from sklearn.manifold import TSNE

from reid import ReIDModel, distance_matrix, list_aliases, load_identity_folders
from reid.data.mot import generate_mot_patches
from reid.eval import ReassociationProtocol, RetrievalProtocol

## Download MOT17

MOT17 is a multi-object tracking benchmark: short sequences where every pedestrian has a box and an identity in every frame. We download its validation half, the frames and the ground-truth boxes.

In [ ]:
MOT17_URL = "https://storage.googleapis.com/com-roboflow-marketing/trackers/datasets/mot17-v2"

!wget -q {MOT17_URL}/mot17-val-frames.zip {MOT17_URL}/mot17-val-annotations.zip
!unzip -qo mot17-val-frames.zip
!unzip -qo mot17-val-annotations.zip

MOT17_VAL = Path("mot17/val")
print(sorted(path.name for path in MOT17_VAL.iterdir()))

## Load a model

`ReIDModel.from_pretrained` takes a model name, a Hugging Face repo, or a local checkpoint. With no argument it loads the default model, OSNet trained on MSMT17. `list_aliases` lists the names you can pass.

In [ ]:
print(list_aliases())

model = ReIDModel.from_pretrained()
print(model.preprocessing.describe())

## Embed the pedestrians in a frame

`extract_embeddings` takes a frame and the boxes in it as `sv.Detections`, the same object a detector returns. Here the boxes come from the MOT17 ground truth, so we also know who is who: we keep their identities in `tracker_id`. MOT17 also boxes pedestrians hidden behind others, so we keep only the ones at least half visible.

In [ ]:
SEQUENCE = MOT17_VAL / "MOT17-02-FRCNN"

# Columns: frame, id, x, y, w, h, active, class, visibility. Class 1 is pedestrian.
ground_truth = np.loadtxt(SEQUENCE / "gt" / "gt.txt", delimiter=",")
ground_truth = ground_truth[(ground_truth[:, 6] == 1) & (ground_truth[:, 7] == 1) & (ground_truth[:, 8] >= 0.5)]


def people_at(frame_index):
    frame = cv2.imread(str(SEQUENCE / "img1" / f"{frame_index:06d}.jpg"))
    rows = ground_truth[ground_truth[:, 0] == frame_index]
    xyxy = sv.clip_boxes(sv.xywh_to_xyxy(rows[:, 2:6]), (frame.shape[1], frame.shape[0]))
    return frame, sv.Detections(xyxy=xyxy, tracker_id=rows[:, 1].astype(int))


frame, detections = people_at(1)
embeddings = model.extract_embeddings(detections, frame)
print(f"{len(detections)} pedestrians -> embeddings of shape {embeddings.shape}")

annotated = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK).annotate(frame.copy(), detections)
annotated = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK).annotate(
    annotated, detections, labels=[f"#{tracker_id}" for tracker_id in detections.tracker_id]
)
sv.plot_image(annotated)

## Match them 30 frames later

A tracker has to match each box to the right identity again after a gap. `distance_matrix` gives the distance from every pedestrian in frame 1 to every pedestrian in frame 31, the same distances `evaluate` ranks with. For each one in frame 1 we pick the closest in frame 31, and check with the ground truth whether it is the same identity. People who left the scene by frame 31 have no correct match, so we leave them out.

In [ ]:
frame_later, detections_later = people_at(31)
embeddings_later = model.extract_embeddings(detections_later, frame_later)

distances = distance_matrix(embeddings, embeddings_later)  # one row per person in frame 1
closest = distances.argmin(axis=1)
matched_ids = detections_later.tracker_id[closest]

in_both = np.isin(detections.tracker_id, detections_later.tracker_id)
correct = matched_ids == detections.tracker_id
print(f"{correct[in_both].sum()} of {in_both.sum()} pedestrians matched to the same identity 30 frames later")

Each pair shows a person in frame 1 (left) and the person the model picked as closest in frame 31 (right), titled with both identities.

In [ ]:
pairs, titles = [], []
for i in np.flatnonzero(in_both):
    before = sv.crop_image(frame, detections.xyxy[i])
    after = sv.crop_image(frame_later, detections_later.xyxy[closest[i]])
    pairs.append(np.hstack([sv.letterbox_image(crop, (64, 128), color=sv.Color.WHITE) for crop in (before, after)]))
    titles.append(f"#{detections.tracker_id[i]} → #{matched_ids[i]}" + ("" if correct[i] else " (wrong)"))

sv.plot_images_grid(pairs, grid_size=(-(-len(pairs) // 6), 6), titles=titles, size=(12, 6))

## Look at the embedding space

Embeddings have 512 dimensions, so to look at them we squeeze them into 2 with t-SNE, which keeps embeddings that are close together close on the plot. We embed everyone in every 10th frame of the video, and color the 10 pedestrians seen most often. Crops of one identity should form a cluster, and identities the model confuses end up near each other.

In [ ]:
sampled = [people_at(frame_index) for frame_index in range(1, int(ground_truth[:, 0].max()) + 1, 10)]
sampled_embeddings = np.concatenate([model.extract_embeddings(d, f) for f, d in sampled])
sampled_ids = np.concatenate([d.tracker_id for _, d in sampled])

points = TSNE(n_components=2, metric="cosine", random_state=0).fit_transform(sampled_embeddings)

ids, counts = np.unique(sampled_ids, return_counts=True)
plt.figure(figsize=(8, 8))
plt.scatter(*points.T, color="lightgray", s=10, label="other people")
for person_id in ids[np.argsort(-counts)[:10]]:
    plt.scatter(*points[sampled_ids == person_id].T, s=16, label=f"#{person_id}")
plt.legend(fontsize=8)
plt.axis("off")
plt.show()

## Cut the video into identity folders

To evaluate a model we need many crops with known identities. `generate_mot_patches` crops every ground-truth box of a MOT-format sequence into one folder per identity, named `<sequence>_<id>`. `load_identity_folders` reads the folders back as a `ReIDSplit`: the image paths, and an identity for each image.

In [ ]:
if not Path("crops").is_dir():
    generate_mot_patches(
        MOT17_VAL,
        "crops",
        split="full",
        sequences=["MOT17-02-FRCNN", "MOT17-09-FRCNN"],
        class_ids=[1],
    )

crops = load_identity_folders("crops")
print(f"{len(crops.image_paths)} crops of {len(np.unique(crops.pids))} people")

`extract_embeddings_from_paths` embeds crops saved on disk, for when you already have them as files.

In [ ]:
print(model.extract_embeddings_from_paths(crops.image_paths[:8]).shape)

## Evaluate with retrieval

`model.evaluate` takes the crops and a protocol, which decides what gets compared. `RetrievalProtocol` takes one crop of each person as the query and ranks all the other crops by distance to it. `rank1` is the percentage of queries whose closest crop is the same person, and `map` measures how high all their correct matches land.

In [ ]:
retrieval = model.evaluate(crops, protocol=RetrievalProtocol())
print(retrieval.metrics)

## Evaluate with re-association

On video, retrieval is easy: the closest crop to a query is usually the same person one frame away. A tracker asks a harder question: which of the boxes 30 frames later has the same identity as this one? `ReassociationProtocol` asks exactly that, ranking only the boxes present `gap_frames` later in the same sequence. `mrr` averages 1 / rank of the correct match.

In [ ]:
reassociation = model.evaluate(crops, protocol=ReassociationProtocol(gap_frames=30))
print(reassociation.metrics)

## Compare models

A protocol picks the same queries every time, so two models are scored on identical inputs. `fastreid_mot17_sbs50` was trained on crops from the other half of these MOT17 videos, the training half.

In [ ]:
protocol = ReassociationProtocol(gap_frames=30)

for name in list_aliases():
    result = ReIDModel.from_pretrained(name).evaluate(crops, protocol=protocol, verbose=False)
    print(f"{name:32} rank1 {result.metrics['rank1']:5.1f}  mrr {result.metrics['mrr']:5.1f}")